# E-commerce Customer Behavior Analysis

**Course:** AIN-375L Data Visualization Lab  
**Student Name:** Nouman Asghar  
**Roll No:** 23238  
**Class:** AI-FA-23  
**Instructor:** Abdul Baqi Malik  

---

This project looks at e-commerce customer data to find patterns in buying behavior. We clean the data, make charts, group customers, and find useful insights.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

## 2. Load Dataset

In [ ]:
df = pd.read_csv('dataset/ecommerce_data.csv')
print('Dataset shape:', df.shape)
print('Columns:', list(df.columns))
df.head(10)

In [ ]:
df.info()

## 3. Data Preprocessing and Cleaning

We need to check for missing values, duplicates, and outliers. Then we make some new columns for better analysis.

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Total missing:', df.isnull().sum().sum())

In [ ]:
df = df.fillna(method='ffill')
print('After filling missing values:', df.isnull().sum().sum())

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
df = df.drop_duplicates()
print('Shape after removing duplicates:', df.shape)

In [ ]:
Q1 = df['total_amount'].quantile(0.25)
Q3 = df['total_amount'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df['total_amount'] < lower) | (df['total_amount'] > upper)]
print(f'IQR: {IQR}')
print(f'Lower bound: {lower}')
print(f'Upper bound: {upper}')
print(f'Outliers found: {len(outliers)}')

df = df[(df['total_amount'] >= lower) & (df['total_amount'] <= upper)]
print(f'Shape after removing outliers: {df.shape}')

### Feature Engineering

Making new columns from existing data to help with analysis.

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])
df['registration_date'] = pd.to_datetime(df['registration_date'])

df['order_month'] = df['order_date'].dt.month
df['order_year'] = df['order_date'].dt.year
df['day_of_week'] = df['order_date'].dt.day_name()
df['order_month_name'] = df['order_date'].dt.month_name()

bins = [17, 25, 35, 45, 55, 66]
labels = ['18-25', '26-35', '36-45', '46-55', '56-65']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels)

df['spending_level'] = pd.cut(df['total_amount'],
    bins=[0, 3000, 8000, 15000, 60000],
    labels=['Low', 'Medium', 'High', 'Very High'])

print('New columns added.')
df[['order_date', 'order_month', 'day_of_week', 'age_group', 'spending_level']].head()

## 4. Exploratory Data Analysis (EDA)

Looking at basic stats and patterns in the data using Pandas and NumPy.

In [ ]:
print('Basic Statistics:')
df.describe()

In [ ]:
print('Gender count:')
print(df['gender'].value_counts())
print()
print('Payment method count:')
print(df['payment_method'].value_counts())
print()
print('Device type count:')
print(df['device_type'].value_counts())
print()
print('Top 5 cities by orders:')
print(df['city'].value_counts().head())

In [ ]:
print('Sales by category:')
cat_sales = df.groupby('product_category')['total_amount'].agg(['sum', 'mean', 'count'])
cat_sales.columns = ['total_sales', 'avg_order', 'num_orders']
cat_sales = cat_sales.sort_values('total_sales', ascending=False)
cat_sales

In [ ]:
print('Sales by city:')
city_sales = df.groupby('city')['total_amount'].agg(['sum', 'mean', 'count'])
city_sales.columns = ['total_sales', 'avg_order', 'num_orders']
city_sales = city_sales.sort_values('total_sales', ascending=False)
city_sales

In [ ]:
numeric_cols = df.select_dtypes(include='number')
corr = numeric_cols.corr()
print('Correlation matrix:')
corr

## 5. Visualizations

We create 17 different charts to understand the data better. Each chart has a title, labels, and an explanation.

### Visualization 1: Bar Chart - Sales by Product Category

In [ ]:
cat_data = df.groupby('product_category')['total_amount'].sum().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
bars = plt.bar(cat_data.index, cat_data.values, color='#4ECDC4', edgecolor='black', linewidth=0.8)
plt.title('Total Sales by Product Category', fontsize=14, fontweight='bold')
plt.xlabel('Product Category')
plt.ylabel('Total Sales (PKR)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('visualizations/01_bar_category_sales.png', dpi=150)
plt.show()

**Insight:** This chart shows which product categories make the most money. Electronics and Home & Kitchen are at the top. Businesses should keep more stock in these categories.

### Visualization 2: Horizontal Bar - Revenue by City

In [ ]:
city_rev = df.groupby('city')['total_amount'].sum().sort_values()

plt.figure(figsize=(10, 6))
plt.barh(city_rev.index, city_rev.values, color='#FF6B9D', edgecolor='black', linewidth=0.8)
plt.title('Revenue by City', fontsize=14, fontweight='bold')
plt.xlabel('Total Revenue (PKR)')
plt.ylabel('City')
plt.tight_layout()
plt.savefig('visualizations/02_hbar_city_revenue.png', dpi=150)
plt.show()

**Insight:** Cities like Karachi and Lahore bring in the most revenue. Marketing efforts should focus more on these big cities.

### Visualization 3: Line Chart - Monthly Sales Trend

In [ ]:
monthly = df.set_index('order_date').resample('M')['total_amount'].sum()

plt.figure(figsize=(14, 5))
plt.plot(monthly.index, monthly.values, marker='o', color='#4ECDC4', linewidth=2, markersize=5)
plt.fill_between(monthly.index, monthly.values, alpha=0.15, color='#4ECDC4')
plt.title('Monthly Sales Trend', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Total Sales (PKR)')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('visualizations/03_line_monthly_sales.png', dpi=150)
plt.show()

**Insight:** Sales go up and down over months. Some months have higher sales which could be due to seasonal shopping or special events.

### Visualization 4: Line Chart - Weekly Order Count

In [ ]:
weekly = df.set_index('order_date').resample('W')['order_id'].count()

plt.figure(figsize=(14, 5))
plt.plot(weekly.index, weekly.values, color='#A855F7', linewidth=1.5, alpha=0.8)
rolling_avg = weekly.rolling(4).mean()
plt.plot(rolling_avg.index, rolling_avg.values, color='#FF6B9D', linewidth=2.5, label='4-Week Avg')
plt.title('Weekly Order Count with Moving Average', fontsize=14, fontweight='bold')
plt.xlabel('Week')
plt.ylabel('Number of Orders')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('visualizations/04_line_weekly_orders.png', dpi=150)
plt.show()

**Insight:** The moving average smooths out the noise. We can see the overall trend of orders over time. It helps spot if the business is growing or not.

### Visualization 5: Heatmap - Correlation Matrix

In [ ]:
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, cmap='YlOrRd', fmt='.2f', linewidths=0.5,
            mask=mask, square=True, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap of Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations/05_heatmap_correlation.png', dpi=150)
plt.show()

**Insight:** The heatmap shows how different numbers relate to each other. Unit price and total amount have a strong positive link. Quantity also affects total amount. Age has little effect on spending.

### Visualization 6: Scatter Plot - Age vs Total Spending

In [ ]:
cust_spend = df.groupby('customer_id').agg({'age': 'first', 'total_amount': 'sum'}).reset_index()

plt.figure(figsize=(10, 6))
plt.scatter(cust_spend['age'], cust_spend['total_amount'], alpha=0.5,
            color='#A855F7', edgecolors='black', s=50, linewidth=0.5)
plt.title('Customer Age vs Total Spending', fontsize=14, fontweight='bold')
plt.xlabel('Age')
plt.ylabel('Total Spending (PKR)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('visualizations/06_scatter_age_spending.png', dpi=150)
plt.show()

**Insight:** Spending is spread across all age groups. There is no clear pattern that older people spend more or less. The business should target all ages equally.

### Visualization 7: Scatter Plot - Session Duration vs Amount

In [ ]:
sample = df.sample(500, random_state=42)

plt.figure(figsize=(10, 6))
plt.scatter(sample['session_duration_min'], sample['total_amount'], alpha=0.4,
            color='#FF6B9D', edgecolors='black', s=40, linewidth=0.5)
plt.title('Session Duration vs Amount Spent', fontsize=14, fontweight='bold')
plt.xlabel('Session Duration (minutes)')
plt.ylabel('Amount Spent (PKR)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('visualizations/07_scatter_session_amount.png', dpi=150)
plt.show()

**Insight:** People who spend more time on the site do not always buy more. Session duration alone is not a good way to predict spending.

### Visualization 8: Pie Chart - Payment Method Distribution

In [ ]:
pay_counts = df['payment_method'].value_counts()
colors = ['#FFD700', '#FF6B9D', '#4ECDC4', '#A855F7', '#FF8C42']

plt.figure(figsize=(8, 8))
wedges, texts, pcts = plt.pie(pay_counts, labels=pay_counts.index, autopct='%1.1f%%',
    colors=colors, edgecolor='black', startangle=90, textprops={'fontsize': 10})
plt.title('Payment Method Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations/08_pie_payment.png', dpi=150)
plt.show()

**Insight:** Cash on Delivery and JazzCash are popular payment methods. The platform should make sure these options work well for customers.

### Visualization 9: Pie Chart - Gender Distribution

In [ ]:
gender_counts = df['gender'].value_counts()

plt.figure(figsize=(7, 7))
plt.pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%',
    colors=['#4ECDC4', '#FF6B9D'], edgecolor='black', startangle=90,
    textprops={'fontsize': 12}, explode=[0.03, 0.03])
plt.title('Customer Gender Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations/09_pie_gender.png', dpi=150)
plt.show()

**Insight:** The gender split is close to equal. Both male and female customers are active buyers. Campaigns should address both genders.

### Visualization 10: Box Plot - Price by Category

In [ ]:
plt.figure(figsize=(14, 6))
order = df.groupby('product_category')['unit_price'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='product_category', y='unit_price', order=order, palette='Set2')
plt.title('Unit Price Distribution by Category', fontsize=14, fontweight='bold')
plt.xlabel('Product Category')
plt.ylabel('Unit Price (PKR)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('visualizations/10_box_price_category.png', dpi=150)
plt.show()

**Insight:** Electronics has the widest price range and highest prices. Books and Food have lower prices. This helps in setting pricing strategies.

### Visualization 11: Box Plot - Satisfaction by Payment Method

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='payment_method', y='satisfaction_score', palette='Pastel1')
plt.title('Satisfaction Score by Payment Method', fontsize=14, fontweight='bold')
plt.xlabel('Payment Method')
plt.ylabel('Satisfaction Score (1-5)')
plt.tight_layout()
plt.savefig('visualizations/11_box_satisfaction_payment.png', dpi=150)
plt.show()

**Insight:** Satisfaction scores are similar across payment methods. No single payment method causes much lower satisfaction. All methods work fine for customers.

### Visualization 12: Distribution Plot - Age

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['age'], kde=True, color='#A855F7', bins=20, edgecolor='black', linewidth=0.5)
plt.axvline(df['age'].mean(), color='red', linestyle='--', label=f"Mean: {df['age'].mean():.1f}")
plt.title('Age Distribution of Customers', fontsize=14, fontweight='bold')
plt.xlabel('Age')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.savefig('visualizations/12_dist_age.png', dpi=150)
plt.show()

**Insight:** Customer ages are spread from 18 to 65. The distribution is fairly even. The platform serves all age groups well.

### Visualization 13: Distribution Plot - Order Amount

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['total_amount'], kde=True, color='#4ECDC4', bins=30, edgecolor='black', linewidth=0.5)
plt.axvline(df['total_amount'].mean(), color='red', linestyle='--', label=f"Mean: {df['total_amount'].mean():.0f} PKR")
plt.axvline(df['total_amount'].median(), color='orange', linestyle='--', label=f"Median: {df['total_amount'].median():.0f} PKR")
plt.title('Distribution of Order Amounts', fontsize=14, fontweight='bold')
plt.xlabel('Total Amount (PKR)')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.savefig('visualizations/13_dist_amount.png', dpi=150)
plt.show()

**Insight:** Most orders are in the lower price range. The distribution is right-skewed meaning fewer large orders. Offering deals on mid-range items could help increase average order size.

### Visualization 14: Violin Plot - Spending by Gender

In [ ]:
plt.figure(figsize=(8, 6))
sns.violinplot(data=df, x='gender', y='total_amount', palette=['#4ECDC4', '#FF6B9D'], inner='box')
plt.title('Spending Distribution by Gender', fontsize=14, fontweight='bold')
plt.xlabel('Gender')
plt.ylabel('Total Amount (PKR)')
plt.tight_layout()
plt.savefig('visualizations/14_violin_gender_spending.png', dpi=150)
plt.show()

**Insight:** Male and female customers have a similar spending pattern. Both groups tend to buy more in the lower price range. Gender is not a major factor in spending.

### Visualization 15: Count Plot - Orders by Day of Week

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='day_of_week', order=day_order, palette='viridis')
plt.title('Number of Orders by Day of Week', fontsize=14, fontweight='bold')
plt.xlabel('Day')
plt.ylabel('Number of Orders')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('visualizations/15_count_dayofweek.png', dpi=150)
plt.show()

**Insight:** Orders are spread across all days. Some days may have slightly more orders. This helps plan delivery schedules and marketing campaigns for specific days.

### Visualization 16: Stacked Bar - Category Sales by Gender

In [ ]:
pivot = df.pivot_table(values='total_amount', index='product_category',
                       columns='gender', aggfunc='sum')

pivot.plot(kind='bar', stacked=True, figsize=(12, 6),
           color=['#4ECDC4', '#FF6B9D'], edgecolor='black', linewidth=0.5)
plt.title('Category Sales by Gender', fontsize=14, fontweight='bold')
plt.xlabel('Product Category')
plt.ylabel('Total Sales (PKR)')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Gender')
plt.tight_layout()
plt.savefig('visualizations/16_stacked_category_gender.png', dpi=150)
plt.show()

**Insight:** Both genders buy from all categories. Some categories like Beauty may lean more toward females. This is useful for targeted ads.

### Visualization 17: Bar Chart - Orders by Device Type

In [ ]:
device_counts = df['device_type'].value_counts()

plt.figure(figsize=(8, 5))
plt.bar(device_counts.index, device_counts.values,
        color=['#FFD700', '#4ECDC4', '#A855F7'], edgecolor='black', linewidth=0.8)
for i, v in enumerate(device_counts.values):
    plt.text(i, v + 10, str(v), ha='center', fontweight='bold')
plt.title('Orders by Device Type', fontsize=14, fontweight='bold')
plt.xlabel('Device')
plt.ylabel('Number of Orders')
plt.tight_layout()
plt.savefig('visualizations/17_bar_device.png', dpi=150)
plt.show()

**Insight:** Mobile is the most used device for shopping. The website must be mobile-friendly. Desktop users are also important but tablets are less common.

## 6. Customer Segmentation using K-Means Clustering

We group customers into segments based on their buying behavior using the K-Means algorithm.

In [ ]:
cust_data = df.groupby('customer_id').agg({
    'total_amount': 'sum',
    'order_id': 'count',
    'session_duration_min': 'mean',
    'satisfaction_score': 'mean'
}).reset_index()

cust_data.columns = ['customer_id', 'total_spending', 'num_orders', 'avg_session', 'avg_satisfaction']
print('Customer data shape:', cust_data.shape)
cust_data.head()

In [ ]:
features = cust_data[['total_spending', 'num_orders', 'avg_session']]
scaler = StandardScaler()
scaled = scaler.fit_transform(features)

inertias = []
K = range(2, 9)
for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(K), inertias, 'bo-', linewidth=2, markersize=8)
plt.title('Elbow Method - Finding Best K', fontsize=14, fontweight='bold')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('visualizations/18_elbow_method.png', dpi=150)
plt.show()

The elbow point is around K=3. We will use 3 clusters.

In [ ]:
km = KMeans(n_clusters=3, random_state=42, n_init=10)
cust_data['cluster'] = km.fit_predict(scaled)

colors_map = {0: '#FF6B9D', 1: '#4ECDC4', 2: '#FFD700'}
cluster_colors = cust_data['cluster'].map(colors_map)

plt.figure(figsize=(10, 7))
for i in range(3):
    mask = cust_data['cluster'] == i
    plt.scatter(cust_data[mask]['total_spending'], cust_data[mask]['num_orders'],
                c=colors_map[i], label=f'Cluster {i}', s=70, edgecolors='black',
                linewidth=0.5, alpha=0.7)

plt.title('Customer Segments (K-Means Clustering)', fontsize=14, fontweight='bold')
plt.xlabel('Total Spending (PKR)')
plt.ylabel('Number of Orders')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('visualizations/19_kmeans_clusters.png', dpi=150)
plt.show()

In [ ]:
print('Cluster Summary:')
summary = cust_data.groupby('cluster')[['total_spending', 'num_orders', 'avg_session', 'avg_satisfaction']].mean()
summary = summary.round(2)
summary

**Insight:** 
- **Cluster 0:** Medium spenders with average orders. These are regular customers.
- **Cluster 1:** Low spenders with fewer orders. These might be new or inactive customers.
- **Cluster 2:** High spenders with many orders. These are the most valuable customers.

The business should reward Cluster 2 with loyalty programs and try to convert Cluster 1 into active buyers.

## 7. Time Series Analysis

Looking at how sales change over time at different levels: daily, weekly, and monthly.

In [ ]:
daily = df.set_index('order_date').resample('D')['total_amount'].sum()
rolling7 = daily.rolling(window=7).mean()
rolling30 = daily.rolling(window=30).mean()

plt.figure(figsize=(14, 6))
plt.plot(daily.index, daily.values, alpha=0.3, color='#4ECDC4', label='Daily Sales')
plt.plot(rolling7.index, rolling7.values, color='#FF6B9D', linewidth=2, label='7-Day Rolling Avg')
plt.plot(rolling30.index, rolling30.values, color='#A855F7', linewidth=2.5, label='30-Day Rolling Avg')
plt.title('Daily Sales with Rolling Averages', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Sales (PKR)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('visualizations/20_timeseries_daily.png', dpi=150)
plt.show()

**Insight:** The 30-day rolling average shows the long-term trend. Daily sales have a lot of ups and downs but the average stays fairly stable.

In [ ]:
monthly_rev = df.set_index('order_date').resample('M')['total_amount'].sum()

plt.figure(figsize=(12, 5))
plt.plot(monthly_rev.index, monthly_rev.values, marker='s', color='#A855F7',
         linewidth=2, markersize=8, markerfacecolor='white', markeredgecolor='#A855F7', markeredgewidth=2)
plt.title('Monthly Revenue Trend', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Revenue (PKR)')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('visualizations/21_timeseries_monthly.png', dpi=150)
plt.show()

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_sales = df.groupby('day_of_week')['total_amount'].mean().reindex(day_order)

plt.figure(figsize=(10, 5))
bars = plt.bar(day_sales.index, day_sales.values, color='#FFD700', edgecolor='black', linewidth=0.8)
plt.title('Average Sales by Day of Week', fontsize=14, fontweight='bold')
plt.xlabel('Day')
plt.ylabel('Average Sales (PKR)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('visualizations/22_timeseries_dayofweek.png', dpi=150)
plt.show()

**Insight:** Some days have higher average sales than others. This pattern helps plan when to run promotions and when staff is needed most.

## 8. Statistical Analysis

Using probability distributions and statistical tests to understand customer behavior patterns.

In [ ]:
amount = df['total_amount']
mu = amount.mean()
sigma = amount.std()

plt.figure(figsize=(10, 6))
sns.histplot(amount, kde=False, stat='density', bins=30, color='#4ECDC4',
             edgecolor='black', alpha=0.7, linewidth=0.5)
x = np.linspace(amount.min(), amount.max(), 200)
plt.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2,
         label=f'Normal Dist (mean={mu:.0f}, std={sigma:.0f})')
plt.title('Spending Distribution with Normal Fit', fontsize=14, fontweight='bold')
plt.xlabel('Amount (PKR)')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.savefig('visualizations/23_stat_normal_fit.png', dpi=150)
plt.show()

**Insight:** The spending data does not follow a perfect normal distribution. It is skewed to the right. This means most orders are small but a few are very large.

In [ ]:
df['z_score'] = np.abs(stats.zscore(df['total_amount']))
outlier_count = (df['z_score'] > 3).sum()
print(f'Z-Score outliers (z > 3): {outlier_count}')
print(f'Percentage: {outlier_count/len(df)*100:.2f}%')
print()

stat_val, p_val = stats.shapiro(df['total_amount'].sample(500, random_state=42))
print(f'Shapiro-Wilk Test for Normality:')
print(f'Statistic = {stat_val:.4f}, p-value = {p_val:.6f}')
if p_val < 0.05:
    print('Result: Data is NOT normally distributed (p < 0.05)')
else:
    print('Result: Data follows normal distribution (p >= 0.05)')

In [ ]:
contingency = pd.crosstab(df['gender'], df['discount_applied'])
chi2, p, dof, expected = stats.chi2_contingency(contingency)

print('Chi-Square Test: Gender vs Discount Usage')
print(f'Chi2 = {chi2:.4f}')
print(f'p-value = {p:.4f}')
print(f'Degrees of freedom = {dof}')
print()
if p < 0.05:
    print('Result: There IS a significant link between gender and discount usage.')
else:
    print('Result: There is NO significant link between gender and discount usage.')

print()
print('Contingency Table:')
contingency

In [ ]:
print('Detailed Statistics:')
stats_table = df[['age', 'unit_price', 'quantity', 'total_amount',
                   'satisfaction_score', 'session_duration_min']].describe()
stats_table.round(2)

**Insight:** The statistical tests tell us important things. The spending data is not normal which is common in real sales data. The chi-square test shows if gender affects discount usage.

## 9. Interactive Visualizations (Plotly)

These charts are interactive. You can hover, zoom, and click on them.

In [ ]:
fig = px.scatter(df.sample(500, random_state=42),
    x='age', y='total_amount', color='product_category',
    size='quantity', hover_data=['customer_name', 'city', 'payment_method'],
    title='Age vs Spending by Category (Interactive)',
    labels={'total_amount': 'Total Amount (PKR)', 'age': 'Customer Age'})
fig.update_layout(height=500)
fig.show()

In [ ]:
cat_rev = df.groupby('product_category')['total_amount'].sum().reset_index()
cat_rev = cat_rev.sort_values('total_amount', ascending=True)

fig = px.bar(cat_rev, x='total_amount', y='product_category', orientation='h',
    color='total_amount', color_continuous_scale='Tealgrn',
    title='Revenue by Category (Interactive)',
    labels={'total_amount': 'Total Revenue (PKR)', 'product_category': 'Category'})
fig.update_layout(height=450)
fig.show()

In [ ]:
fig = make_subplots(rows=2, cols=2,
    subplot_titles=('Monthly Revenue', 'Top Categories', 'Payment Methods', 'Age Groups'),
    specs=[[{'type': 'scatter'}, {'type': 'bar'}],
           [{'type': 'pie'}, {'type': 'bar'}]])

monthly_data = df.set_index('order_date').resample('M')['total_amount'].sum()
fig.add_trace(go.Scatter(x=monthly_data.index, y=monthly_data.values,
    mode='lines+markers', name='Monthly Revenue',
    line=dict(color='#4ECDC4', width=2)), row=1, col=1)

cat_data = df.groupby('product_category')['total_amount'].sum().sort_values(ascending=False).head(5)
fig.add_trace(go.Bar(x=cat_data.index, y=cat_data.values,
    name='Top 5 Categories', marker_color='#FF6B9D'), row=1, col=2)

pay_data = df['payment_method'].value_counts()
fig.add_trace(go.Pie(labels=pay_data.index, values=pay_data.values,
    name='Payments'), row=2, col=1)

age_data = df.groupby('age_group')['total_amount'].mean()
fig.add_trace(go.Bar(x=age_data.index.astype(str), y=age_data.values,
    name='Avg Spending by Age', marker_color='#A855F7'), row=2, col=2)

fig.update_layout(height=700, title_text='E-commerce Dashboard Overview', showlegend=False)
fig.show()

**Insight:** The interactive dashboard lets you explore data by hovering and zooming. It combines multiple views in one place for quick analysis.

## 10. Pair Plot - Multi Feature Relationships

In [ ]:
pair_cols = df[['age', 'unit_price', 'quantity', 'total_amount', 'satisfaction_score']].sample(300, random_state=42)

g = sns.pairplot(pair_cols, diag_kind='kde',
    plot_kws={'alpha': 0.5, 'edgecolor': 'black', 's': 20, 'linewidth': 0.3})
g.figure.suptitle('Feature Relationships (Pair Plot)', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations/24_pairplot.png', dpi=100)
plt.show()

**Insight:** The pair plot shows relationships between all numeric features at once. Unit price and total amount are strongly linked. Other features show weaker connections.

## 11. Key Insights and Recommendations

### Main Findings

1. **Top categories:** Electronics and Home & Kitchen make the most sales. Stock these well.

2. **City performance:** Bigger cities like Karachi and Lahore bring more revenue. Focus marketing there.

3. **Customer segments:** K-Means found 3 groups - high spenders, regular buyers, and low spenders. Each needs a different approach.

4. **Payment preferences:** Cash on Delivery and mobile payments (JazzCash, EasyPaisa) are popular. Keep these options available.

5. **Mobile first:** Most orders come from mobile devices. The website must work great on phones.

6. **Discounts work:** Customers who get discounts tend to have higher satisfaction scores.

7. **Age is not a factor:** All age groups spend about the same. Do not limit marketing to one age group.

8. **Sales are right-skewed:** Most orders are small. Offering bundle deals could increase average order value.

### Recommendations

- **For high spenders:** Give them VIP status and early access to new products.
- **For low spenders:** Send targeted emails with small discounts to bring them back.
- **For all customers:** Make the mobile app faster and easier to use.
- **For inventory:** Keep more stock in Electronics and Home & Kitchen.
- **For marketing:** Run campaigns on weekdays when order volume is highest.

---

**End of Analysis**